In [ ]:
# Input: Species_abundance.xlsx, sheet Merged, with samples in rows and species abundance columns matching the model features.
# Input: serialized model dictionaries in Models; species names must match feature_columns. Optional sample identifiers: index, Metaphlan_ID, Sample_ID, Full_name.

from pathlib import Path
import json
import re
import sys
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

PROJECT_ROOT = Path.cwd()
INPUT_XLSX = PROJECT_ROOT / 'Species_abundance.xlsx'
SHEET_NAME = 'Merged'
MODEL_DIR = PROJECT_ROOT / 'models'
if not MODEL_DIR.is_dir():
    MODEL_DIR = PROJECT_ROOT / 'Models'
OUTPUT_DIR = PROJECT_ROOT / 'shap_relative_abundance_outputs'
TARGETS = ['TA', 'IAM', 'ILA', 'IAld', 'IE', 'IAA', 'IA', 'IPA', 'Indole']
MAX_DISPLAY = 25
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def safe_filename(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value))


In [ ]:
if not INPUT_XLSX.exists():
    raise FileNotFoundError(INPUT_XLSX)
if not MODEL_DIR.exists():
    raise FileNotFoundError(MODEL_DIR)

data = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)
if data.empty:
    raise ValueError('Input table is empty')
if data.columns.duplicated().any():
    raise ValueError('Input table contains duplicated column names')

sample_id_column = next((name for name in ['index', 'Metaphlan_ID', 'Sample_ID', 'Full_name'] if name in data.columns), None)
if sample_id_column is None:
    sample_ids = pd.Series(np.arange(len(data)), index=data.index, name='row_number')
else:
    sample_ids = data[sample_id_column].copy()
    if sample_ids.isna().any():
        raise ValueError(f'{sample_id_column} contains missing values')
    if sample_ids.astype(str).duplicated().any():
        raise ValueError(f'{sample_id_column} contains duplicated values')

MODEL_TARGET_ALIASES = {'I3C': 'IAld', 'INDOLE': 'Indole'}
saved_models = {}
for model_path in sorted(MODEL_DIR.glob('*.joblib')):
    artifact = joblib.load(model_path)
    if not isinstance(artifact, dict):
        raise TypeError(f'{model_path.name} is not a model dictionary')
    target = MODEL_TARGET_ALIASES.get(artifact.get('target'), artifact.get('target'))
    if target not in TARGETS:
        raise ValueError(f'Unexpected model target: {target}')
    if target in saved_models:
        raise ValueError(f'Multiple saved models found for target: {target}')
    saved_models[target] = (model_path, artifact)

def load_model(target):
    if target not in saved_models:
        raise FileNotFoundError(f'No saved model found for target: {target}')
    model_path, artifact = saved_models[target]
    required_keys = {'target', 'model', 'feature_columns', 'x_transform', 'y_transform', 'training_n'}
    missing_keys = sorted(required_keys - set(artifact))
    if missing_keys:
        raise ValueError(f'{model_path.name} is missing keys: {missing_keys}')
    if artifact['x_transform'] != 'log2(x + 1)':
        raise ValueError(f'{target} does not use log2(x + 1) for inputs')
    if artifact['y_transform'] != 'log2(y + 1)':
        raise ValueError(f'{target} does not use log2(y + 1) for outputs')
    if not hasattr(artifact['model'], 'predict'):
        raise TypeError(f'{target} model has no predict method')
    return model_path, artifact

models = {}
model_audit_rows = []
for target in TARGETS:
    model_path, artifact = load_model(target)
    feature_columns = list(artifact['feature_columns'])
    if len(feature_columns) != len(set(feature_columns)):
        raise ValueError(f'{target} contains duplicated feature names')
    n_features_in = getattr(artifact['model'], 'n_features_in_', None)
    if n_features_in is not None and int(n_features_in) != len(feature_columns):
        raise ValueError(f'{target} has an inconsistent feature count')
    models[target] = {'path': model_path, 'artifact': artifact}
    model_audit_rows.append({
        'Target': target,
        'Model_File': model_path.name,
        'Model_Name': artifact.get('model_name'),
        'Artifact_Target': artifact.get('target'),
        'Estimator_Class': type(artifact['model']).__name__,
        'Training_N': int(artifact['training_n']),
        'N_Features': len(feature_columns)
    })

declared_feature_union = list(dict.fromkeys(
    feature
    for target in TARGETS
    for feature in models[target]['artifact']['feature_columns']
))
model_audit = pd.DataFrame(model_audit_rows)
print(f'Samples: {len(data)}')
print(f'Input columns: {data.shape[1]}')
print(f'Model features: {len(declared_feature_union)}')
display(model_audit)


In [ ]:
def as_nonnegative_numeric(series, name):
    numeric = pd.to_numeric(series, errors='coerce')
    invalid = series.notna() & numeric.isna()
    if invalid.any():
        values = series.loc[invalid].astype(str).head(5).tolist()
        raise ValueError(f'{name} contains non-numeric values: {values}')
    if not np.isfinite(numeric.dropna().to_numpy(dtype=float)).all():
        raise ValueError(f'{name} contains infinite values')
    if (numeric < 0).any():
        raise ValueError(f'{name} contains negative values')
    return numeric.astype(float)

if not set(declared_feature_union).intersection(data.columns):
    raise ValueError('No input species columns match the saved model features. Check table orientation and species names.')

raw_values = {}
missing_training_features = []
for feature in declared_feature_union:
    if feature in data.columns:
        raw_values[feature] = as_nonnegative_numeric(data[feature], feature)
    else:
        raw_values[feature] = pd.Series(0.0, index=data.index)
        missing_training_features.append(feature)

raw_union = pd.DataFrame(raw_values, index=data.index)
missing_input_cells = int(raw_union.isna().sum().sum())
raw_union = raw_union.fillna(0.0)
if not np.isfinite(raw_union.to_numpy(dtype=float)).all():
    raise ValueError('Input data contains invalid values')

X_log2_union = pd.DataFrame(
    np.log2(raw_union.to_numpy(dtype=float) + 1.0),
    index=raw_union.index,
    columns=raw_union.columns
)
sample_ids = sample_ids.astype(str).to_numpy()
print(f'Missing feature columns filled with zero: {len(missing_training_features)}')
print(f'Missing input cells filled with zero: {missing_input_cells:,}')


In [ ]:
def extract_shap_values(explanation, n_rows, n_features):
    values = np.asarray(explanation.values, dtype=float)
    if values.ndim == 2 and values.shape == (n_rows, n_features):
        return values
    if values.ndim == 3 and values.shape == (n_rows, n_features, 1):
        return values[:, :, 0]
    raise ValueError(f'Unsupported SHAP array shape: {values.shape}')

def extract_base_values(explanation, n_rows):
    values = np.asarray(explanation.base_values, dtype=float)
    if values.ndim == 0:
        return np.full(n_rows, float(values))
    if values.ndim == 1 and values.shape[0] == n_rows:
        return values
    if values.ndim == 2 and values.shape == (n_rows, 1):
        return values[:, 0]
    if values.size == 1:
        return np.full(n_rows, float(values.reshape(-1)[0]))
    raise ValueError(f'Unsupported base_values shape: {values.shape}')

all_importance_rows = []
all_audit_rows = []
for target in TARGETS:
    artifact = models[target]['artifact']
    model = artifact['model']
    feature_columns = list(artifact['feature_columns'])
    X_model = X_log2_union.loc[:, feature_columns].copy()
    explainer = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent')
    explanation = explainer(X_model)
    shap_values = extract_shap_values(explanation, len(X_model), len(feature_columns))
    base_values = extract_base_values(explanation, len(X_model))
    pred_log2 = np.asarray(model.predict(X_model), dtype=float).reshape(-1)
    pred_reconstructed = base_values + shap_values.sum(axis=1)
    additivity_error = float(np.max(np.abs(pred_reconstructed - pred_log2)))
    if not np.isfinite(shap_values).all() or not np.isfinite(pred_log2).all():
        raise ValueError(f'{target} has invalid SHAP values or predictions')
    pred_original = np.maximum(np.power(2.0, pred_log2) - 1.0, 0.0)
    mean_abs = np.abs(shap_values).mean(axis=0)
    mean_signed = shap_values.mean(axis=0)
    positive_fraction = (shap_values > 0).mean(axis=0)
    nonzero_fraction = (np.abs(shap_values) > 1e-12).mean(axis=0)
    ranks = pd.Series(mean_abs, index=feature_columns).rank(method='first', ascending=False).astype(int).to_numpy()
    for feature, rank, abs_value, signed_value, positive_value, nonzero_value in zip(feature_columns, ranks, mean_abs, mean_signed, positive_fraction, nonzero_fraction):
        all_importance_rows.append({
            'Target': target,
            'Rank': int(rank),
            'Feature': feature,
            'MeanAbsSHAP_Log2Target': float(abs_value),
            'MeanSHAP_Log2Target': float(signed_value),
            'PositiveSHAP_Fraction': float(positive_value),
            'NonzeroSHAP_Fraction': float(nonzero_value)
        })
    safe_target = safe_filename(target)
    importance_one = pd.DataFrame([row for row in all_importance_rows if row['Target'] == target]).sort_values(['MeanAbsSHAP_Log2Target', 'Feature'], ascending=[False, True]).reset_index(drop=True)
    importance_one['Rank'] = np.arange(1, len(importance_one) + 1)
    importance_one.to_csv(OUTPUT_DIR / f'shap_importance_{safe_target}.csv', index=False, encoding='utf-8-sig')
    prediction_one = pd.DataFrame({
        'Sample_ID': sample_ids,
        'Target': target,
        'Predicted_log2_y': pred_log2,
        'Predicted_y_original_scale': pred_original
    })
    prediction_one.to_csv(OUTPUT_DIR / f'predictions_{safe_target}.csv', index=False, encoding='utf-8-sig')
    shap_one = pd.DataFrame(shap_values, columns=feature_columns)
    shap_one.insert(0, 'Sample_ID', sample_ids)
    shap_one.insert(1, 'base_value_log2_y', base_values)
    shap_one.insert(2, 'model_output_log2_y', pred_log2)
    shap_one.insert(3, 'model_output_reconstructed_log2_y', pred_reconstructed)
    shap_one.insert(4, 'predicted_y_original_scale', pred_original)
    shap_one.to_csv(OUTPUT_DIR / f'shap_values_{safe_target}_log2_target.csv', index=False, encoding='utf-8-sig')
    plt.close('all')
    shap.summary_plot(shap_values, X_model, feature_names=feature_columns, plot_type='bar', max_display=MAX_DISPLAY, show=False, plot_size=(10, 8))
    plt.title(f'{target}: mean(|SHAP|), output = log2(y + 1)')
    plt.xlabel('mean(|SHAP value|) on log2(y + 1) scale')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'shap_bar_{safe_target}.pdf', format='pdf', bbox_inches='tight')
    plt.close('all')
    shap.summary_plot(shap_values, X_model, feature_names=feature_columns, cmap='viridis', max_display=MAX_DISPLAY, show=False, plot_size=(11, 9))
    plt.title(f'{target}: SHAP beeswarm, output = log2(y + 1)')
    plt.xlabel('SHAP value on log2(y + 1) scale')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'shap_beeswarm_{safe_target}.pdf', format='pdf', bbox_inches='tight')
    plt.close('all')
    all_audit_rows.append({
        'Target': target,
        'Model_File': models[target]['path'].name,
        'Input_N': len(data),
        'Training_N_recorded': int(artifact['training_n']),
        'N_Features': len(feature_columns),
        'SHAP_Additivity_MaxAbsError_Log2': additivity_error
    })
    print(f'{target}: {importance_one.iloc[0]["Feature"]}')

importance_all = pd.DataFrame(all_importance_rows).sort_values(['Target', 'MeanAbsSHAP_Log2Target', 'Feature'], ascending=[True, False, True]).reset_index(drop=True)
importance_all['Rank'] = importance_all.groupby('Target').cumcount() + 1
audit_all = pd.DataFrame(all_audit_rows)
importance_all.to_csv(OUTPUT_DIR / 'shap_importance_all_targets.csv', index=False, encoding='utf-8-sig')
audit_all.to_csv(OUTPUT_DIR / 'shap_run_audit.csv', index=False, encoding='utf-8-sig')
model_audit.to_csv(OUTPUT_DIR / 'model_audit.csv', index=False, encoding='utf-8-sig')
manifest = {
    'input_file': str(INPUT_XLSX),
    'sheet_name': SHEET_NAME,
    'targets': TARGETS,
    'input_n': int(len(data)),
    'feature_union_n': int(len(declared_feature_union)),
    'x_processing': 'missing values and absent features to 0; log2(x + 1)',
    'shap_output_scale': 'log2(y + 1)',
    'shap_method': "TreeExplainer(feature_perturbation='tree_path_dependent')",
    'image_format': 'pdf',
    'font_family': 'Arial',
    'python': sys.version,
    'shap_version': shap.__version__,
    'joblib_version': joblib.__version__
}
with open(OUTPUT_DIR / 'shap_run_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)
print(f'Output directory: {OUTPUT_DIR}')
display(importance_all.groupby('Target', sort=False).head(20).reset_index(drop=True))
display(audit_all)
